In this file, the embedding from DLPFC data will be generated. SOMDE will be used to select the highly variable genes. If you don't have the data, Please run the somdeProcess.ipynb to generate the data. 

In [1]:
"""
Compute gene cost matrices using precomputed SOMDE genes.

Assumes:
- SOMDE results already exist as CSV files
- Only gene cost computation is performed here

Author: RAFT-UP
"""

import os
import torch
import scanpy as sc
import pandas as pd

from _gene_cost_somde_updated_0122 import compute_gene_cost


# --------------------------------------------------
# Configuration
# --------------------------------------------------
ROOT_DIR = "/home/salovjade/0324_raftupdata/DLPFC12"
SOMDE_DIR = "./data4train/somde_updated_0122"
OUT_DIR = "./outputEmbed_updated_0122"

N_TOP_SV_GENES = 3000
DEVICE = "cuda:0"


# SECTION_PAIRS = [
#     ("151507", "151508"),
#     ("151508", "151509"),
#     ("151509", "151510")
# ]

SECTION_PAIRS = [
    ("151669", "151670"),
    ("151670", "151671"),
    ("151671", "151672"),
    ("151673", "151674"),
    ("151674", "151675"),
    ("151675", "151676")
]




# --------------------------------------------------
# Data loader (Visium)
# --------------------------------------------------
def load_DLPFC(root_dir: str, section_id: str):
    """
    Load one DLPFC Visium slice with ground-truth layer labels.
    """
    adata = sc.read_visium(
        path=os.path.join(root_dir, section_id),
        count_file=f"{section_id}_filtered_feature_bc_matrix.h5",
    )
    adata.var_names_make_unique()

    gt_dir = os.path.join(root_dir, section_id, "gt")
    gt_df = pd.read_csv(
        os.path.join(gt_dir, "tissue_positions_list_GTs.txt"),
        sep=",",
        header=None,
        index_col=0,
    )

    adata.obs["original_clusters"] = gt_df.loc[:, 6]
    keep_bcs = adata.obs.dropna().index
    adata = adata[keep_bcs].copy()
    adata.obs["original_clusters"] = (
        adata.obs["original_clusters"].astype(int).astype(str)
    )

    print(f"[load_DLPFC] Loaded section {section_id}")
    return adata


# --------------------------------------------------
# Main loop
# --------------------------------------------------
if __name__ == "__main__":

    os.makedirs(OUT_DIR, exist_ok=True)

    for sid_A, sid_B in SECTION_PAIRS:

        print(f"\n=== Computing gene cost: {sid_A} vs {sid_B} ===")

        # --------------------------------------------------
        # 1. Load raw Visium slices
        # --------------------------------------------------
        sliceA = load_DLPFC(ROOT_DIR, sid_A)
        sliceB = load_DLPFC(ROOT_DIR, sid_B)

        # --------------------------------------------------
        # 2. Expression preprocessing (for representation learning)
        # --------------------------------------------------
        for sl in (sliceA, sliceB):
            sc.pp.normalize_total(sl)
            sc.pp.log1p(sl)

        # --------------------------------------------------
        # 3. Load SOMDE-selected genes
        # --------------------------------------------------
        df_somde_A = pd.read_csv(f"{SOMDE_DIR}/somde_{sid_A}.csv")
        df_somde_B = pd.read_csv(f"{SOMDE_DIR}/somde_{sid_B}.csv")

        sv_genes_A = df_somde_A["g"].values[:N_TOP_SV_GENES]
        sv_genes_B = df_somde_B["g"].values[:N_TOP_SV_GENES]

        sliceA = sliceA[:, sv_genes_A].copy()
        sliceB = sliceB[:, sv_genes_B].copy()

        # --------------------------------------------------
        # 4. Scale  (node features)
        # --------------------------------------------------
        for sl in (sliceA, sliceB):
            sc.pp.scale(sl)

        # --------------------------------------------------
        # 5. Compute gene cost matrix
        # --------------------------------------------------
        compute_gene_cost(
            sliceA=sliceA,
            sliceB=sliceB,
            section_id_A=sid_A,
            section_id_B=sid_B,
            n_h=100,
            n_epoch=3500,
            lr=2e-4,
            print_step=500,
            seed=0,
            device=DEVICE,
            output_dir=OUT_DIR,
        )

    print("\nAll gene cost matrices computed successfully.")


=== Computing gene cost: 151669 vs 151670 ===
[load_DLPFC] Loaded section 151669
[load_DLPFC] Loaded section 151670
[0000] DGI loss = 2.9031
[0500] DGI loss = 0.0051
[1000] DGI loss = 0.0023
[1500] DGI loss = 0.0005
[2000] DGI loss = 0.0003
[2500] DGI loss = 0.0002
[3000] DGI loss = 0.0001
[3499] DGI loss = 0.0001

=== Computing gene cost: 151670 vs 151671 ===
[load_DLPFC] Loaded section 151670
[load_DLPFC] Loaded section 151671
[0000] DGI loss = 2.8499
[0500] DGI loss = 0.0060
[1000] DGI loss = 0.0016
[1500] DGI loss = 0.0004
[2000] DGI loss = 0.0003
[2500] DGI loss = 0.0001
[3000] DGI loss = 0.0001
[3499] DGI loss = 0.0001

=== Computing gene cost: 151671 vs 151672 ===
[load_DLPFC] Loaded section 151671
[load_DLPFC] Loaded section 151672
[0000] DGI loss = 3.0347
[0500] DGI loss = 0.0014
[1000] DGI loss = 0.0003
[1500] DGI loss = 0.0002
[2000] DGI loss = 0.0001
[2500] DGI loss = 0.0001
[3000] DGI loss = 0.0000
[3499] DGI loss = 0.0000

=== Computing gene cost: 151673 vs 151674 ===
[l